In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import random

DATASET_PATH = "/kaggle/input/datasets/daikhiyassine/dataset"
SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]

counts = {split: {} for split in SPLITS}
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(DATASET_PATH, split, cls)
        counts[split][cls] = len(os.listdir(cls_path)) if os.path.isdir(cls_path) else 0

print("Résumé :")
for split in SPLITS:
    total = sum(counts[split].values())
    print(f"\n{split.upper()} (total: {total})")
    for cls in CLASSES:
        print(f"   {cls}: {counts[split][cls]}")

x = np.arange(len(CLASSES))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 6))
for i, split in enumerate(SPLITS):
    values = [counts[split][cls] for cls in CLASSES]
    bars = ax.bar(x + i * width, values, width, label=split)
    ax.bar_label(bars, fontsize=8)
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'images")
ax.set_title("Distribution des classes par split - Dataset Aigis (YOLO26)", fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(CLASSES)
ax.legend(title="Split")
plt.tight_layout()
plt.savefig('/kaggle/working/distribution_classes_yolo26.png', dpi=150, bbox_inches='tight')
plt.show()

n_samples = 4
fig, axes = plt.subplots(len(CLASSES), n_samples, figsize=(n_samples * 3, len(CLASSES) * 3))
for i, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATASET_PATH, "train", cls)
    images = os.listdir(cls_path)
    sample_imgs = random.sample(images, min(n_samples, len(images)))
    for j, img_name in enumerate(sample_imgs):
        img = Image.open(os.path.join(cls_path, img_name))
        ax = axes[i, j]
        ax.imshow(img)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(cls, fontsize=12, fontweight='bold', rotation=90)
        ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=9)
plt.suptitle("Échantillons du dataset (train) par classe", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/echantillons_dataset_yolo26.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
import hashlib
from PIL import Image
from collections import defaultdict
import json

DATASET_PATH = "/kaggle/input/datasets/daikhiyassine/dataset"
CLEAN_PATH = "/kaggle/working/dataset_clean_yolo26"
SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]
SPLIT_PRIORITY = {"train": 0, "val": 1, "test": 2}

# YOLO26 classification utilise 224x224 en entrée par défaut (comme B0)
TARGET_SIZE = 256  # marge pour augmentation, YOLO redimensionnera à 224 en interne
JPEG_QUALITY = 85

print("Étape 1/3 : identification des doublons...")

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_map = defaultdict(list)
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(DATASET_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            h = get_file_hash(img_path)
            hash_map[h].append((split, cls, img_name, img_path))

files_to_skip = set()
leak_count = 0
duplicates = {h: v for h, v in hash_map.items() if len(v) > 1}
for h, entries in duplicates.items():
    entries_sorted = sorted(entries, key=lambda e: SPLIT_PRIORITY[e[0]])
    remove = entries_sorted[1:]
    for split, cls, img_name, img_path in remove:
        files_to_skip.add(img_path)
    if len(set(e[0] for e in entries)) > 1:
        leak_count += 1

print(f"Doublons à exclure : {len(files_to_skip)} | Fuites entre splits : {leak_count}")

print(f"\nÉtape 2/3 : resize à {TARGET_SIZE}x{TARGET_SIZE} + compression JPEG q={JPEG_QUALITY}...")
converted_count = 0
copied_count = 0

# ⚠️ Ultralytics attend une structure : dataset/train/classe/, dataset/val/classe/, dataset/test/classe/
# → structure déjà identique à celle de nos scripts précédents, pas de changement de format nécessaire

for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(DATASET_PATH, split, cls)
        dst_dir = os.path.join(CLEAN_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.isdir(src_dir):
            continue
        for img_name in os.listdir(src_dir):
            src_path = os.path.join(src_dir, img_name)
            if src_path in files_to_skip:
                continue
            base_name = os.path.splitext(img_name)[0]
            dst_path = os.path.join(dst_dir, base_name + ".jpg")
            try:
                with Image.open(src_path) as img:
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                        converted_count += 1
                    img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
                    img.save(dst_path, "JPEG", quality=JPEG_QUALITY, optimize=True)
                    copied_count += 1
            except Exception as e:
                print(f"   ⚠️ Erreur sur {src_path} : {e}")

print(f"\nImages traitées : {copied_count} | Converties RGB : {converted_count}")

print("\nÉtape 3/3 : taille finale...")
def get_dir_size(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total

size_gb = get_dir_size(CLEAN_PATH) / (1024 ** 3)
print(f"Taille finale : {size_gb:.2f} GB")

final_counts = {}
for split in SPLITS:
    final_counts[split] = {cls: len(os.listdir(os.path.join(CLEAN_PATH, split, cls))) for cls in CLASSES}
    print(f"{split}: {final_counts[split]}")

rapport = {
    "modele_cible": "YOLO26 (classification)",
    "target_size": TARGET_SIZE,
    "jpeg_quality": JPEG_QUALITY,
    "doublons_exclus": len(files_to_skip),
    "fuites_entre_splits": leak_count,
    "images_converties_rgb": converted_count,
    "taille_finale_gb": round(size_gb, 2),
    "comptage_final": final_counts
}
with open("/kaggle/working/rapport_nettoyage_final_yolo26.json", "w") as f:
    json.dump(rapport, f, indent=2, ensure_ascii=False)

print(f"\n✅ Dataset nettoyé pour YOLO26 : {CLEAN_PATH}")

In [ ]:
# Installation d'Ultralytics (contient YOLO26)
!pip install -q ultralytics

from ultralytics import YOLO
import os

# ============================================================
# CONFIGURATION
# ============================================================
DATASET_PATH = "/kaggle/working/dataset_clean_yolo26"
IMG_SIZE = 224  # taille standard pour la classification YOLO
EPOCHS_SEARCH = 10  # epochs par essai pendant la recherche (léger, comme Keras Tuner)
ITERATIONS = 15     # nombre de combinaisons testées

# ⚠️ Pour la classification, Ultralytics attend un dossier racine contenant
# directement train/ val/ test/ (déjà notre cas — pas de fichier YAML nécessaire ici,
# contrairement à la détection d'objets)

# ============================================================
# 1️⃣ Chargement du modèle pré-entraîné (nano = le plus rapide pour la recherche)
# ============================================================
model = YOLO("yolo26n-cls.pt")  # nano : idéal pour explorer vite ; on passera à s/m pour l'entraînement final

# ============================================================
# 2️⃣ Recherche d'hyperparamètres (tuning génétique intégré à Ultralytics)
# ============================================================
print("Lancement de la recherche d'hyperparamètres YOLO26...")

results = model.tune(
    data=DATASET_PATH,
    epochs=EPOCHS_SEARCH,
    iterations=ITERATIONS,
    imgsz=IMG_SIZE,
    batch=32,
    optimizer="AdamW",
    plots=False,
    save=False,
    val=True,
    project="/kaggle/working/yolo26_tune",
    name="aigis_yolo26_search",
    exist_ok=True
)

# ============================================================
# 3️⃣ Résultats : meilleurs hyperparamètres trouvés
# ============================================================
import yaml

best_hp_path = "/kaggle/working/yolo26_tune/aigis_yolo26_search/best_hyperparameters.yaml"

if os.path.exists(best_hp_path):
    with open(best_hp_path, "r") as f:
        best_hyperparameters = yaml.safe_load(f)

    print("\n" + "=" * 60)
    print("MEILLEURS HYPERPARAMÈTRES TROUVÉS - YOLO26")
    print("=" * 60)
    for k, v in best_hyperparameters.items():
        print(f"   {k}: {v}")

    import json
    with open("/kaggle/working/best_hyperparameters_yolo26.json", "w") as f:
        json.dump(best_hyperparameters, f, indent=2)

    print(f"\n✅ Sauvegardé dans /kaggle/working/best_hyperparameters_yolo26.json")
else:
    print("⚠️ Fichier de résultats non trouvé, vérifie le dossier /kaggle/working/yolo26_tune/")

In [ ]:
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics

In [ ]:
import torch
print("PyTorch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("GPU détecté :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")

# Test rapide d'un calcul sur GPU pour confirmer la compatibilité sm_60
x = torch.rand(1000, 1000).cuda()
y = torch.rand(1000, 1000).cuda()
z = x @ y
print("✅ Calcul GPU réussi, pas d'erreur sm_60")

In [ ]:
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics

In [ ]:
import torch
print("PyTorch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

x = torch.rand(1000, 1000).cuda()
y = torch.rand(1000, 1000).cuda()
z = x @ y
print("✅ Calcul GPU réussi")

In [ ]:
!pip install -q ultralytics

In [ ]:
import os
import hashlib
from PIL import Image
from collections import defaultdict
import json

DATASET_PATH = "/kaggle/input/datasets/daikhiyassine/dataset"
CLEAN_PATH = "/kaggle/working/dataset_clean_yolo26"
SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]
SPLIT_PRIORITY = {"train": 0, "val": 1, "test": 2}

TARGET_SIZE = 256
JPEG_QUALITY = 85

print("Étape 1/3 : identification des doublons...")

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_map = defaultdict(list)
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(DATASET_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            h = get_file_hash(img_path)
            hash_map[h].append((split, cls, img_name, img_path))

files_to_skip = set()
leak_count = 0
duplicates = {h: v for h, v in hash_map.items() if len(v) > 1}
for h, entries in duplicates.items():
    entries_sorted = sorted(entries, key=lambda e: SPLIT_PRIORITY[e[0]])
    remove = entries_sorted[1:]
    for split, cls, img_name, img_path in remove:
        files_to_skip.add(img_path)
    if len(set(e[0] for e in entries)) > 1:
        leak_count += 1

print(f"Doublons à exclure : {len(files_to_skip)} | Fuites entre splits : {leak_count}")

print(f"\nÉtape 2/3 : resize à {TARGET_SIZE}x{TARGET_SIZE} + compression JPEG q={JPEG_QUALITY}...")
converted_count = 0
copied_count = 0

for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(DATASET_PATH, split, cls)
        dst_dir = os.path.join(CLEAN_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.isdir(src_dir):
            continue
        for img_name in os.listdir(src_dir):
            src_path = os.path.join(src_dir, img_name)
            if src_path in files_to_skip:
                continue
            base_name = os.path.splitext(img_name)[0]
            dst_path = os.path.join(dst_dir, base_name + ".jpg")
            try:
                with Image.open(src_path) as img:
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                        converted_count += 1
                    img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
                    img.save(dst_path, "JPEG", quality=JPEG_QUALITY, optimize=True)
                    copied_count += 1
            except Exception as e:
                print(f"   ⚠️ Erreur sur {src_path} : {e}")

print(f"\nImages traitées : {copied_count} | Converties RGB : {converted_count}")
print(f"\n✅ Dataset nettoyé : {CLEAN_PATH}")

In [ ]:
from ultralytics import YOLO
import os

DATASET_PATH = "/kaggle/working/dataset_clean_yolo26"
IMG_SIZE = 224
EPOCHS_SEARCH = 10
ITERATIONS = 15

model = YOLO("yolo26n-cls.pt")

print("Lancement de la recherche d'hyperparamètres YOLO26...")

results = model.tune(
    data=DATASET_PATH,
    epochs=EPOCHS_SEARCH,
    iterations=ITERATIONS,
    imgsz=IMG_SIZE,
    batch=32,
    optimizer="AdamW",
    plots=False,
    save=False,
    val=True,
    project="/kaggle/working/yolo26_tune",
    name="aigis_yolo26_search",
    exist_ok=True
)

import yaml
best_hp_path = "/kaggle/working/yolo26_tune/aigis_yolo26_search/best_hyperparameters.yaml"

if os.path.exists(best_hp_path):
    with open(best_hp_path, "r") as f:
        best_hyperparameters = yaml.safe_load(f)

    print("\n" + "=" * 60)
    print("MEILLEURS HYPERPARAMÈTRES TROUVÉS - YOLO26")
    print("=" * 60)
    for k, v in best_hyperparameters.items():
        print(f"   {k}: {v}")

    import json
    with open("/kaggle/working/best_hyperparameters_yolo26.json", "w") as f:
        json.dump(best_hyperparameters, f, indent=2)

    print(f"\n✅ Sauvegardé dans /kaggle/working/best_hyperparameters_yolo26.json")
else:
    print("⚠️ Fichier de résultats non trouvé, vérifie le dossier /kaggle/working/yolo26_tune/")

In [ ]:
from ultralytics import YOLO
import json
import os

# ============================================================
# CONFIGURATION
# ============================================================
DATASET_PATH = "/kaggle/working/dataset_clean_yolo26"
IMG_SIZE = 224
EPOCHS = 50  # entraînement complet, plus long que la recherche (10 epochs/trial)
BATCH_SIZE = 32

# Charge les meilleurs hyperparamètres trouvés
with open("/kaggle/working/best_hyperparameters_yolo26.json") as f:
    best_hp = json.load(f)

print("Hyperparamètres utilisés :", best_hp)

# ============================================================
# 1️⃣ Chargement du modèle
# ============================================================
# On utilise yolo26s-cls (small) pour l'entraînement final — plus de capacité
# que le nano utilisé pendant la recherche, tout en restant léger
model = YOLO("yolo26s-cls.pt")

# ============================================================
# 2️⃣ Entraînement avec les meilleurs hyperparamètres
# ============================================================
print("\n" + "=" * 60)
print("ENTRAÎNEMENT FINAL - YOLO26s Classification")
print("=" * 60)

results = model.train(
    data=DATASET_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    optimizer="AdamW",
    patience=10,  # early stopping si pas d'amélioration pendant 10 epochs
    project="/kaggle/working/yolo26_train",
    name="aigis_yolo26_final",
    exist_ok=True,
    plots=True,
    save=True,
    val=True,

    # Hyperparamètres issus du tuning
    lr0=best_hp["lr0"],
    lrf=best_hp["lrf"],
    momentum=best_hp["momentum"],
    weight_decay=best_hp["weight_decay"],
    warmup_epochs=best_hp["warmup_epochs"],
    warmup_momentum=best_hp["warmup_momentum"],
    box=best_hp["box"],
    cls=best_hp["cls"],
    dfl=best_hp["dfl"],
    hsv_h=best_hp["hsv_h"],
    hsv_s=best_hp["hsv_s"],
    hsv_v=best_hp["hsv_v"],
    degrees=best_hp["degrees"],
    translate=best_hp["translate"],
    scale=best_hp["scale"],
    shear=best_hp["shear"],
    perspective=best_hp["perspective"],
    flipud=best_hp["flipud"],
    fliplr=best_hp["fliplr"],
    mosaic=best_hp["mosaic"],
    mixup=best_hp["mixup"],
    close_mosaic=int(best_hp["close_mosaic"]),
)

# ============================================================
# 3️⃣ Sauvegarde finale + export
# ============================================================
best_model_path = "/kaggle/working/yolo26_train/aigis_yolo26_final/weights/best.pt"
print(f"\n✅ Meilleur modèle sauvegardé : {best_model_path}")

# Export en TFLite pour déploiement mobile/embarqué (comme pour B0/B2)
model_best = YOLO(best_model_path)
model_best.export(format="tflite", imgsz=IMG_SIZE)
print("✅ Modèle exporté en TFLite")

In [ ]:
import torch
print("PyTorch version :", torch.__version__)

In [ ]:
import os
print("Dataset existe :", os.path.exists("/kaggle/working/dataset_clean_yolo26"))
print("Hyperparamètres existent :", os.path.exists("/kaggle/working/best_hyperparameters_yolo26.json"))

In [ ]:
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics

In [ ]:
# Vérification PyTorch
import torch
print("PyTorch version :", torch.__version__)
x = torch.rand(1000, 1000).cuda()
y = torch.rand(1000, 1000).cuda()
z = x @ y
print("✅ Calcul GPU réussi\n")

# Nettoyage du dataset
import os
import hashlib
from PIL import Image
from collections import defaultdict
import json

DATASET_PATH = "/kaggle/input/datasets/daikhiyassine/dataset"
CLEAN_PATH = "/kaggle/working/dataset_clean_yolo26"
SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]
SPLIT_PRIORITY = {"train": 0, "val": 1, "test": 2}
TARGET_SIZE = 256
JPEG_QUALITY = 85

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_map = defaultdict(list)
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(DATASET_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            h = get_file_hash(img_path)
            hash_map[h].append((split, cls, img_name, img_path))

files_to_skip = set()
for h, entries in hash_map.items():
    if len(entries) > 1:
        entries_sorted = sorted(entries, key=lambda e: SPLIT_PRIORITY[e[0]])
        for split, cls, img_name, img_path in entries_sorted[1:]:
            files_to_skip.add(img_path)

for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(DATASET_PATH, split, cls)
        dst_dir = os.path.join(CLEAN_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.isdir(src_dir):
            continue
        for img_name in os.listdir(src_dir):
            src_path = os.path.join(src_dir, img_name)
            if src_path in files_to_skip:
                continue
            base_name = os.path.splitext(img_name)[0]
            dst_path = os.path.join(dst_dir, base_name + ".jpg")
            try:
                with Image.open(src_path) as img:
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                    img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
                    img.save(dst_path, "JPEG", quality=JPEG_QUALITY, optimize=True)
            except Exception as e:
                print(f"⚠️ Erreur sur {src_path} : {e}")

print(f"✅ Dataset nettoyé : {CLEAN_PATH}")

# Restauration des hyperparamètres déjà trouvés (pas besoin de refaire le tuning)
best_hp = {
    "lr0": 0.00056, "lrf": 0.01, "momentum": 0.9008, "weight_decay": 0.00098,
    "warmup_epochs": 1.92603, "warmup_momentum": 0.9497, "box": 11.85996,
    "cls": 0.46957, "cls_pw": 0.00013, "dfl": 2.95113, "hsv_h": 0.00581,
    "hsv_s": 0.81219, "hsv_v": 0.32101, "degrees": 0.01077, "translate": 0.11043,
    "scale": 0.4843, "shear": 0.00319, "perspective": 0.00089, "flipud": 0.00154,
    "fliplr": 0.28429, "bgr": 0.00207, "mosaic": 0.8527, "mixup": 0.00037,
    "cutmix": 0.00223, "copy_paste": 0.00566, "close_mosaic": 10
}
with open("/kaggle/working/best_hyperparameters_yolo26.json", "w") as f:
    json.dump(best_hp, f, indent=2)
print("✅ Hyperparamètres restaurés")

In [ ]:
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics

In [ ]:
# Vérification PyTorch
import torch
print("PyTorch version :", torch.__version__)
x = torch.rand(1000, 1000).cuda()
y = torch.rand(1000, 1000).cuda()
z = x @ y
print("✅ Calcul GPU réussi\n")

# Nettoyage du dataset
import os
import hashlib
from PIL import Image
from collections import defaultdict
import json

DATASET_PATH = "/kaggle/input/datasets/daikhiyassine/dataset"
CLEAN_PATH = "/kaggle/working/dataset_clean_yolo26"
SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]
SPLIT_PRIORITY = {"train": 0, "val": 1, "test": 2}
TARGET_SIZE = 256
JPEG_QUALITY = 85

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_map = defaultdict(list)
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(DATASET_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            h = get_file_hash(img_path)
            hash_map[h].append((split, cls, img_name, img_path))

files_to_skip = set()
for h, entries in hash_map.items():
    if len(entries) > 1:
        entries_sorted = sorted(entries, key=lambda e: SPLIT_PRIORITY[e[0]])
        for split, cls, img_name, img_path in entries_sorted[1:]:
            files_to_skip.add(img_path)

for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(DATASET_PATH, split, cls)
        dst_dir = os.path.join(CLEAN_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.isdir(src_dir):
            continue
        for img_name in os.listdir(src_dir):
            src_path = os.path.join(src_dir, img_name)
            if src_path in files_to_skip:
                continue
            base_name = os.path.splitext(img_name)[0]
            dst_path = os.path.join(dst_dir, base_name + ".jpg")
            try:
                with Image.open(src_path) as img:
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                    img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
                    img.save(dst_path, "JPEG", quality=JPEG_QUALITY, optimize=True)
            except Exception as e:
                print(f"⚠️ Erreur sur {src_path} : {e}")

print(f"✅ Dataset nettoyé : {CLEAN_PATH}")

# Restauration des hyperparamètres déjà trouvés (pas besoin de refaire le tuning)
best_hp = {
    "lr0": 0.00056, "lrf": 0.01, "momentum": 0.9008, "weight_decay": 0.00098,
    "warmup_epochs": 1.92603, "warmup_momentum": 0.9497, "box": 11.85996,
    "cls": 0.46957, "cls_pw": 0.00013, "dfl": 2.95113, "hsv_h": 0.00581,
    "hsv_s": 0.81219, "hsv_v": 0.32101, "degrees": 0.01077, "translate": 0.11043,
    "scale": 0.4843, "shear": 0.00319, "perspective": 0.00089, "flipud": 0.00154,
    "fliplr": 0.28429, "bgr": 0.00207, "mosaic": 0.8527, "mixup": 0.00037,
    "cutmix": 0.00223, "copy_paste": 0.00566, "close_mosaic": 10
}
with open("/kaggle/working/best_hyperparameters_yolo26.json", "w") as f:
    json.dump(best_hp, f, indent=2)
print("✅ Hyperparamètres restaurés")

In [ ]:
import ultralytics
print(ultralytics.__version__)

In [ ]:
from ultralytics import YOLO
import json
import os

DATASET_PATH = "/kaggle/working/dataset_clean_yolo26"
IMG_SIZE = 224
EPOCHS = 50
BATCH_SIZE = 32

with open("/kaggle/working/best_hyperparameters_yolo26.json") as f:
    best_hp = json.load(f)

print("Hyperparamètres utilisés :", best_hp)

model = YOLO("yolo26s-cls.pt")

print("\n" + "=" * 60)
print("ENTRAÎNEMENT FINAL - YOLO26s Classification")
print("=" * 60)

results = model.train(
    data=DATASET_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    optimizer="AdamW",
    patience=10,
    project="/kaggle/working/yolo26_train",
    name="aigis_yolo26_final",
    exist_ok=True,
    plots=True,
    save=True,
    val=True,
    lr0=best_hp["lr0"],
    lrf=best_hp["lrf"],
    momentum=best_hp["momentum"],
    weight_decay=best_hp["weight_decay"],
    warmup_epochs=best_hp["warmup_epochs"],
    warmup_momentum=best_hp["warmup_momentum"],
    box=best_hp["box"],
    cls=best_hp["cls"],
    dfl=best_hp["dfl"],
    hsv_h=best_hp["hsv_h"],
    hsv_s=best_hp["hsv_s"],
    hsv_v=best_hp["hsv_v"],
    degrees=best_hp["degrees"],
    translate=best_hp["translate"],
    scale=best_hp["scale"],
    shear=best_hp["shear"],
    perspective=best_hp["perspective"],
    flipud=best_hp["flipud"],
    fliplr=best_hp["fliplr"],
    mosaic=best_hp["mosaic"],
    mixup=best_hp["mixup"],
    close_mosaic=int(best_hp["close_mosaic"]),
)

best_model_path = "/kaggle/working/yolo26_train/aigis_yolo26_final/weights/best.pt"
print(f"\n✅ Meilleur modèle sauvegardé : {best_model_path}")

model_best = YOLO(best_model_path)
model_best.export(format="tflite", imgsz=IMG_SIZE)
print("✅ Modèle exporté en TFLite")

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

DATASET_PATH = "/kaggle/working/dataset_clean_yolo26"
MODEL_PATH = "/kaggle/working/yolo26_train/aigis_yolo26_final/weights/best.pt"
IMG_SIZE = 224

model = YOLO(MODEL_PATH)

print("Évaluation native Ultralytics sur le split test...")
metrics = model.val(
    data=DATASET_PATH,
    split="test",
    imgsz=IMG_SIZE,
    project="/kaggle/working/yolo26_test",
    name="aigis_yolo26_test",
    exist_ok=True,
    plots=True
)

print(f"\nTop-1 Accuracy : {metrics.top1:.4f}")
print(f"Top-5 Accuracy : {metrics.top5:.4f}")

test_dir = os.path.join(DATASET_PATH, "test")
class_names = sorted(os.listdir(test_dir))
print("Classes :", class_names)

y_true, y_pred, y_pred_probs = [], [], []

for cls_idx, cls_name in enumerate(class_names):
    cls_dir = os.path.join(test_dir, cls_name)
    for img_name in os.listdir(cls_dir):
        img_path = os.path.join(cls_dir, img_name)
        result = model.predict(img_path, imgsz=IMG_SIZE, verbose=False)[0]
        probs = result.probs.data.cpu().numpy()
        pred_class = int(np.argmax(probs))
        y_true.append(cls_idx)
        y_pred.append(pred_class)
        y_pred_probs.append(probs)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_probs = np.array(y_pred_probs)

print("\n" + "=" * 60)
print("RAPPORT DE CLASSIFICATION - YOLO26")
print("=" * 60)
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

y_true_onehot = np.eye(len(class_names))[y_true]
auc_macro = roc_auc_score(y_true_onehot, y_pred_probs, average="macro", multi_class="ovr")
print(f"AUC macro (sklearn) : {auc_macro:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Prédiction"); plt.ylabel("Vraie classe")
plt.title("Matrice de confusion - YOLO26 (test set)")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_yolo26.png", dpi=150, bbox_inches="tight")
plt.show()

cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Oranges", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Prédiction"); plt.ylabel("Vraie classe")
plt.title("Matrice de confusion normalisée (%) - YOLO26")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_normalized_yolo26.png", dpi=150, bbox_inches="tight")
plt.show()

results_dict = {
    "top1_accuracy": float(metrics.top1),
    "top5_accuracy": float(metrics.top5),
    "auc_macro_sklearn": float(auc_macro),
    "classification_report": classification_report(y_true, y_pred, target_names=class_names, output_dict=True),
    "confusion_matrix": cm.tolist()
}
with open("/kaggle/working/test_results_yolo26.json", "w") as f:
    json.dump(results_dict, f, indent=2)

print("\n✅ Résultats sauvegardés")

In [ ]:
model.export(format="onnx", imgsz=IMG_SIZE)